In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.paths import RAW_DIR

raw_path = RAW_DIR / "v_crg_student_course_raw.parquet"
df = pd.read_parquet(raw_path)

df_plot = df.copy()

# تحويل part_id إلى رقم مع إجبار القيم الخاطئة أن تصبح NaN
df_plot["part_id_num"] = pd.to_numeric(df_plot["part_id"], errors="coerce")

# تحويله إلى نص بعد إزالة .0
df_plot["part_id_clean"] = (
    df_plot["part_id_num"]
    .astype("Int64")       # يدعم القيم الفارغة
    .astype("string")
)

# نحتفظ فقط بالقيم الصحيحة التي طولها 5 مثل 20191
valid_part_mask = df_plot["part_id_clean"].str.len().eq(5)

df_plot_valid = df_plot[valid_part_mask].copy()

# استخراج السنة والفصل
df_plot_valid["year"] = df_plot_valid["part_id_clean"].str[:4].astype(int)
df_plot_valid["semester"] = df_plot_valid["part_id_clean"].str[4].astype(int)

# إنشاء عمود واضح مثل 2019-S1
df_plot_valid["year_semester"] = (
    df_plot_valid["year"].astype(str)
    + "-S"
    + df_plot_valid["semester"].astype(str)
)

df_plot_valid = df_plot_valid.sort_values(["year", "semester"])


In [ ]:
plt.figure(figsize=(14, 6))

order = (
    df_plot_valid[["year", "semester", "year_semester"]]
    .drop_duplicates()
    .sort_values(["year", "semester"])["year_semester"]
)

sns.countplot(
    data=df_plot_valid,
    x="year_semester",
    order=order
)

plt.title("Distribution of Records by Academic Semester")
plt.xlabel("Academic Semester")
plt.ylabel("Number of Records")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
invalid_part_ids = df_plot[~valid_part_mask]

print("Total rows:", len(df_plot))
print("Valid part_id rows:", len(df_plot_valid))
print("Invalid or missing part_id rows:", len(invalid_part_ids))

invalid_part_ids["part_id"].value_counts(dropna=False).head(20) 


In [ ]:
pivot_table = df_plot_valid.pivot_table(
    index="year",
    columns="semester",
    values="part_id",
    aggfunc="count",
    fill_value=0
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    pivot_table,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Records Distribution by Year and Semester")
plt.xlabel("Semester")
plt.ylabel("Year")
plt.tight_layout()
plt.show() 